# 6. AIME: AI Adoption Maturity Evaluator

Scores repository AI tool adoption maturity (L1–L4) using three signals:
1. **Tool detection** — known AI tool patterns from `Artifacts/*.json`
2. **Path semantic intent** — embed artifact paths, classify against category templates
3. **Content semantic classification** — existing file embeddings vs category templates

## Maturity Levels

| Level | Name | Description |
|-------|------|-------------|
| L1 | Ad Hoc | No AI artifacts found |
| L2 | Grounded Prompting | Rules, configuration, architecture, code-style files |
| L3 | Agent-Augmented | Agents, commands, skills files |
| L4 | Agentic Orchestration | Flows, session-logs |

**Model**: `nomic-ai/nomic-embed-text-v1.5` (768 dims, prefix `"clustering: "`)

In [ ]:
# IMPORTANT: Set OpenMP environment variables BEFORE any other imports to prevent kernel crash on macOS ARM
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OMP_MAX_ACTIVE_LEVELS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
# ANTHROPIC_API_KEY is loaded from .env via src/__init__.py
# Copy .env.example to .env and set your key there


import sys
import json
import pickle
import gc
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Add project root to path
sys.path.insert(0, os.path.abspath('..'))

from src.embedding_generator import (
    load_embedding_model,
    DEFAULT_TASK_PREFIX,
    DEFAULT_MODEL
)
from src.maturity_scorer import (
    CATEGORY_NAMES,
    CATEGORY_TO_LEVEL,
    MATURITY_LABELS,
    MaturityLevel,
    embed_category_templates,
    classify_by_tool_detection,
    classify_by_path,
    classify_by_content,
    combine_signals,
    aggregate_repo_maturity,
    build_artifacts_map,
    build_tool_category_matrix,
    generate_report,
    score_from_output_dir,
)

print(f"Task prefix: '{DEFAULT_TASK_PREFIX}'")
print(f"Model: {DEFAULT_MODEL}")
print(f"Categories ({len(CATEGORY_NAMES)}): {CATEGORY_NAMES}")

## Configuration

In [ ]:
# Paths
PROJECT_ROOT = Path("..").resolve()
ARTIFACTS_DIR = str(PROJECT_ROOT / "Artifacts")
OUTPUT_DIR = PROJECT_ROOT / "embedding_output"  # Change to "output" for single-repo runs
EXPORT_DIR = PROJECT_ROOT / "data"
EXPORT_DIR.mkdir(exist_ok=True)

# Select repo to evaluate
REPO_NAME = "claude-flow"  # Change to your repo name

# Level colors for consistent charts
LEVEL_COLORS = {
    2: "#3b82f6",  # blue
    3: "#f97316",  # orange
    4: "#22c55e",  # green
}
LEVEL_LABELS = {2: "L2 Grounded", 3: "L3 Agent-Augmented", 4: "L4 Orchestration"}

print(f"Output dir: {OUTPUT_DIR}")
print(f"Repo: {REPO_NAME}")

## Phase 1: Load Model & Embed Templates

In [ ]:
model = load_embedding_model(DEFAULT_MODEL)
template_embeddings = embed_category_templates(model)
print(f"Template embeddings: {template_embeddings.shape}")

## Phase 2: Single Repo Evaluation

In [ ]:
score = score_from_output_dir(
    str(OUTPUT_DIR), REPO_NAME, model,
    artifacts_dir=ARTIFACTS_DIR,
)

print(f"{'='*60}")
print(f"MATURITY ASSESSMENT: {REPO_NAME}")
print(f"{'='*60}")
print(f"Level: L{score.overall_level} — {score.overall_label}")
print(f"Confidence: {score.confidence:.1%}")
print(f"Artifacts: {score.artifact_count}")
print(f"Tools: {', '.join(score.tools_detected) or 'none'}")
print()
print("Level Evidence:")
for lvl in (2, 3, 4):
    ev = score.level_evidence.get(lvl, {})
    p, s = ev.get('primary', 0), ev.get('secondary', 0)
    print(f"  L{lvl}: {p} primary, {s} secondary")
print()
print("Coherence Checks:")
for flag in score.coherence_flags:
    icon = {"green": "✅", "yellow": "⚠️", "red": "❌"}.get(flag.status, "❓")
    print(f"  {icon} {flag.check}: {flag.message}")
print()
print("Recommendations:")
for i, rec in enumerate(score.recommendations, 1):
    print(f"  {i}. {rec}")

In [ ]:
# Build artifacts map
from src.maturity_scorer import FileClassification

# Reconstruct FileClassification objects from the DataFrame
fc_df = score.file_classifications
if fc_df is not None and not fc_df.empty:
    # Build artifacts map from the DataFrame
    artifacts_map = build_artifacts_map([
        FileClassification(
            file_id=str(row.get('file_id', '')),
            artifact_path=str(row.get('artifact_path', '')),
            tool_name=str(row.get('tool_name', 'unknown')),
            discovery_step=str(row.get('discovery_step', '')),
            content_primary=row.get('content_primary'),
            content_primary_score=float(row.get('content_primary_score', 0)),
            path_primary=row.get('path_primary'),
            path_primary_score=float(row.get('path_primary_score', 0)),
            signals_agree=bool(row.get('signals_agree', False)),
            assigned_category=row.get('assigned_category'),
            assigned_maturity_level=row.get('assigned_maturity_level'),
            categories_within_threshold=(
                str(row.get('categories_within_threshold', '')).split('+')
                if row.get('categories_within_threshold') else []
            ),
        )
        for _, row in fc_df.iterrows()
    ])
    print("Artifacts Map:")
    display(artifacts_map)
else:
    print("No file classifications available.")
    artifacts_map = pd.DataFrame()

## Phase 2.5: Temporal Health Analysis

Joins per-artifact git commit history with AIME file classifications to produce lifecycle classifications, monthly activity heatmaps, and per-category health verdicts.

In [ ]:
import importlib
import src.temporal_health
importlib.reload(src.temporal_health)
from src.temporal_health import analyze_temporal_health

temporal = analyze_temporal_health(
    repo_dir=str(OUTPUT_DIR / REPO_NAME),
    file_classifications=score.file_classifications,
    repo_name=REPO_NAME,
)

if temporal.has_timeseries:
    print(f"Timeseries loaded: {temporal.artifact_count} of {temporal.total_classified} "
          f"classified artifacts have git commit history")
    print(f"Date range: {temporal.earliest_date} to {temporal.horizon_date}")
    print(f"Categories with temporal data: {list(temporal.author_diversity.keys())}")
    print()
    print("Lifecycle distribution:")
    if not temporal.category_summaries.empty:
        display(temporal.category_summaries)
    print()
    print("Health verdicts:")
    for v in temporal.health_verdicts:
        icon = {"healthy": "✅", "warning": "⚠️", "concern": "❌"}.get(v["verdict"], "❓")
        print(f"  {icon} {v['message']}")
else:
    print("No timeseries CSV found — temporal analysis skipped.")

## Phase 3: Visualizations

### Chart 1: Maturity Level Gauge

In [ ]:
fig = go.Figure(go.Indicator(
    mode="gauge+number+delta",
    value=score.overall_level,
    title={"text": f"{REPO_NAME}<br><span style='font-size:0.7em;color:gray'>Confidence: {score.confidence:.0%}</span>"},
    number={"prefix": "L", "font": {"size": 60}},
    gauge={
        "axis": {"range": [0, 4], "tickvals": [1, 2, 3, 4], "ticktext": ["L1", "L2", "L3", "L4"]},
        "bar": {"color": LEVEL_COLORS.get(score.overall_level, "#6b7280")},
        "steps": [
            {"range": [0, 1], "color": "#f3f4f6"},
            {"range": [1, 2], "color": "#dbeafe"},
            {"range": [2, 3], "color": "#ffedd5"},
            {"range": [3, 4], "color": "#dcfce7"},
        ],
        "threshold": {
            "line": {"color": "red", "width": 4},
            "thickness": 0.75,
            "value": score.overall_level,
        },
    },
))
fig.update_layout(height=350, width=500)
fig_gauge = fig
fig.show()

### Chart 2: Level Stacking Bar (Primary + Secondary Evidence)

In [ ]:
levels = [2, 3, 4]
primary_counts = [score.level_evidence.get(lvl, {}).get('primary', 0) for lvl in levels]
secondary_counts = [score.level_evidence.get(lvl, {}).get('secondary', 0) for lvl in levels]
level_names = [LEVEL_LABELS[lvl] for lvl in levels]

fig = go.Figure()
fig.add_trace(go.Bar(
    y=level_names, x=primary_counts, orientation='h',
    name='Primary', marker_color=[LEVEL_COLORS[lvl] for lvl in levels],
    text=primary_counts, textposition='auto',
))
fig.add_trace(go.Bar(
    y=level_names, x=secondary_counts, orientation='h',
    name='Secondary (embedded)', marker_color=[LEVEL_COLORS[lvl] for lvl in levels],
    marker_pattern_shape='/', opacity=0.5,
    text=secondary_counts, textposition='auto',
))
fig.update_layout(
    title='Level Stacking: Primary vs Secondary Evidence',
    barmode='group', height=300, width=800,
    xaxis_title='Artifact count',
    legend=dict(orientation='h', yanchor='bottom', y=1.02),
)
fig_stacking = fig
fig.show()

### Chart 3: Category Distribution (9-Category Breakdown)

In [ ]:
fig_categories = None
if not artifacts_map.empty:
    # Sort categories by level (L4 on top, L2 on bottom)
    sorted_map = artifacts_map.sort_values(
        'maturity_level', ascending=True
    )
    cats = [f"{c} (L{CATEGORY_TO_LEVEL[c]})" for c in sorted_map['category']]
    path_counts = sorted_map['primary_path'].tolist()
    content_counts = sorted_map['primary_content'].tolist()
    colors = [LEVEL_COLORS[CATEGORY_TO_LEVEL[c]] for c in sorted_map['category']]

    fig = go.Figure()
    fig.add_trace(go.Bar(
        y=cats, x=content_counts, orientation='h',
        name='Content-based', marker_color=colors, opacity=0.8,
        text=content_counts, textposition='outside',
    ))
    fig.add_trace(go.Bar(
        y=cats, x=path_counts, orientation='h',
        name='Path-based', marker_color=colors, opacity=0.4,
        marker_pattern_shape='/',
        text=path_counts, textposition='outside',
    ))
    fig.update_layout(
        title='Category Distribution: Content vs Path Classification',
        barmode='group', height=450, width=900,
        xaxis_title='File count',
        legend=dict(orientation='h', yanchor='bottom', y=1.02),
    )
    fig_categories = fig
    fig.show()
else:
    print("No data for category distribution.")

### Chart 4: Signal Agreement Matrix

In [ ]:
fig_agreement = None
if fc_df is not None and not fc_df.empty and 'path_primary' in fc_df.columns and 'content_primary' in fc_df.columns:
    # Filter to rows with both signals
    valid = fc_df.dropna(subset=['path_primary', 'content_primary'])
    if not valid.empty:
        agreement = pd.crosstab(valid['path_primary'], valid['content_primary'])
        # Reorder to CATEGORY_NAMES
        ordered_cats = [c for c in CATEGORY_NAMES if c in agreement.columns or c in agreement.index]
        agreement = agreement.reindex(index=ordered_cats, columns=ordered_cats, fill_value=0)

        fig = go.Figure(go.Heatmap(
            z=agreement.values, x=agreement.columns.tolist(), y=agreement.index.tolist(),
            colorscale='YlOrRd',
            text=agreement.values, texttemplate='%{text}',
            textfont={'size': 10},
        ))
        fig.update_layout(
            title='Signal Agreement: Path Intent (rows) vs Content (columns)',
            xaxis_title='Content top-1', yaxis_title='Path top-1',
            height=500, width=600,
        )
        fig_agreement = fig
        fig.show()

        agreed = fc_df['signals_agree'].sum()
        total = len(fc_df)
        print(f"Agreement rate: {agreed}/{total} ({agreed/total*100:.1f}%)")
    else:
        print("No valid signal pairs for agreement matrix.")
else:
    print("No data for signal agreement matrix.")

### Chart 5: Multi-Category Attribution Sunburst

In [ ]:
fig_sunburst = None
if not artifacts_map.empty:
    sunburst_ids = []
    sunburst_labels = []
    sunburst_parents = []
    sunburst_values = []
    sunburst_colors = []

    for lvl in (2, 3, 4):
        lvl_label = f"L{lvl}"
        lvl_cats = [c for c in CATEGORY_NAMES if CATEGORY_TO_LEVEL[c] == lvl]
        lvl_total = sum(
            artifacts_map.loc[artifacts_map['category'] == c, 'total_primary'].values[0]
            for c in lvl_cats
        )
        if lvl_total > 0:
            sunburst_ids.append(lvl_label)
            sunburst_labels.append(lvl_label)
            sunburst_parents.append("")
            sunburst_values.append(lvl_total)
            sunburst_colors.append(LEVEL_COLORS[lvl])

            for c in lvl_cats:
                count = artifacts_map.loc[artifacts_map['category'] == c, 'total_primary'].values[0]
                if count > 0:
                    sunburst_ids.append(f"{lvl_label}-{c}")
                    sunburst_labels.append(c)
                    sunburst_parents.append(lvl_label)
                    sunburst_values.append(count)
                    sunburst_colors.append(LEVEL_COLORS[lvl])

    if sunburst_ids:
        fig = go.Figure(go.Sunburst(
            ids=sunburst_ids,
            labels=sunburst_labels,
            parents=sunburst_parents,
            values=sunburst_values,
            marker=dict(colors=sunburst_colors),
            branchvalues='total',
        ))
        fig.update_layout(
            title='Maturity Composition Sunburst',
            height=500, width=600,
        )
        fig_sunburst = fig
        fig.show()
    else:
        print("No artifacts to display in sunburst.")
else:
    print("No data for sunburst chart.")

### Chart 6: Tool × Category Heatmap

In [ ]:
fig_tool_cat = None
if fc_df is not None and not fc_df.empty:
    # Reconstruct FileClassification objects for the matrix builder
    fcs_for_matrix = [
        FileClassification(
            file_id=str(row.get('file_id', '')),
            artifact_path=str(row.get('artifact_path', '')),
            tool_name=str(row.get('tool_name', 'unknown')),
            discovery_step='',
            assigned_category=row.get('assigned_category'),
        )
        for _, row in fc_df.iterrows()
    ]
    tool_cat_matrix = build_tool_category_matrix(fcs_for_matrix)

    if not tool_cat_matrix.empty:
        fig = go.Figure(go.Heatmap(
            z=tool_cat_matrix.values,
            x=tool_cat_matrix.columns.tolist(),
            y=tool_cat_matrix.index.tolist(),
            colorscale='Blues',
            text=tool_cat_matrix.values,
            texttemplate='%{text}',
            textfont={'size': 10},
        ))
        fig.update_layout(
            title='Tool × Category Heatmap',
            xaxis_title='Category', yaxis_title='Tool',
            height=max(300, len(tool_cat_matrix) * 40 + 100),
            width=800,
        )
        fig_tool_cat = fig
        fig.show()
    else:
        print("No known-tool artifacts for tool×category matrix.")
else:
    print("No data for tool×category heatmap.")

### Chart 7: Hybrid Score Distribution

In [ ]:
fig_hybrid = None
if fc_df is not None and not fc_df.empty and 'categories_within_threshold' in fc_df.columns:
    # Compute hybrid score from categories_within_threshold
    hybrid_scores = fc_df['categories_within_threshold'].apply(
        lambda x: len(str(x).split('+')) if pd.notna(x) and str(x).strip() else 1
    )

    # Count files per score value and assign colors
    score_counts = hybrid_scores.value_counts().sort_index()
    x_vals = score_counts.index.tolist()
    y_vals = score_counts.values.tolist()

    # Color: green=1 (focused), yellow=2 (moderate), red=3+ (diffuse)
    bar_colors = []
    for sv in x_vals:
        if sv == 1:
            bar_colors.append('#22c55e')   # green — focused
        elif sv == 2:
            bar_colors.append('#eab308')   # yellow — moderate
        else:
            bar_colors.append('#ef4444')   # red — diffuse

    fig = go.Figure(go.Bar(
        x=x_vals, y=y_vals,
        marker_color=bar_colors,
        text=y_vals, textposition='auto',
    ))
    fig.update_layout(
        title='Hybrid Score Distribution (categories within 0.03 of top-1)',
        xaxis_title='Number of categories matched',
        yaxis_title='File count',
        xaxis=dict(dtick=1),
        height=350, width=600,
    )
    fig_hybrid = fig
    fig.show()

    pure = (hybrid_scores == 1).sum()
    dual = (hybrid_scores == 2).sum()
    multi = (hybrid_scores >= 3).sum()
    total = len(hybrid_scores)
    print(f"Focused (1 category): {pure} ({pure/total*100:.1f}%)")
    print(f"Moderate (2 categories): {dual} ({dual/total*100:.1f}%)")
    print(f"Diffuse (3+ categories): {multi} ({multi/total*100:.1f}%)")
else:
    print("No data for hybrid score distribution.")

### Chart 8: Coherence Dashboard

In [ ]:
fig_coherence = None
if score.coherence_flags:
    checks = [f.check for f in score.coherence_flags]
    statuses = [f.status for f in score.coherence_flags]
    messages = [f.message for f in score.coherence_flags]

    status_colors = {'green': '#22c55e', 'yellow': '#eab308', 'red': '#ef4444'}
    colors = [status_colors.get(s, '#6b7280') for s in statuses]
    status_values = {'green': 3, 'yellow': 2, 'red': 1}
    values = [status_values.get(s, 0) for s in statuses]

    fig = go.Figure(go.Bar(
        y=checks, x=values, orientation='h',
        marker_color=colors,
        text=[f"{s.upper()}" for s in statuses],
        textposition='inside',
        hovertext=messages,
        hoverinfo='text',
    ))
    fig.update_layout(
        title='Coherence Dashboard',
        xaxis=dict(tickvals=[1, 2, 3], ticktext=['Red', 'Yellow', 'Green'], range=[0, 3.5]),
        height=max(250, len(checks) * 50 + 100), width=700,
    )
    fig_coherence = fig
    fig.show()
else:
    print("No coherence checks to display (likely L1 — no artifacts).")

### Chart 9: Artifact Lifecycle Classification

In [ ]:
fig_lifecycle_bars = None
if 'temporal' in dir() and temporal.has_timeseries and not temporal.category_summaries.empty:
    cs = temporal.category_summaries.copy()
    # Sort by maturity level
    cs['level'] = cs['category'].map(lambda c: CATEGORY_TO_LEVEL.get(c, 2))
    cs = cs.sort_values('level')

    lifecycle_types = ['steady', 'burst', 'set-and-forget', 'abandoned', 'no-history']
    lifecycle_colors = {
        'steady': '#22c55e',        # green
        'burst': '#eab308',          # amber
        'set-and-forget': '#64748b', # slate
        'abandoned': '#ef4444',      # red
        'no-history': '#d1d5db',     # light gray
    }

    fig = go.Figure()
    for lc in lifecycle_types:
        fig.add_trace(go.Bar(
            y=[f"{c} (L{CATEGORY_TO_LEVEL.get(c, '?')})" for c in cs['category']],
            x=cs[lc],
            orientation='h',
            name=lc,
            marker_color=lifecycle_colors[lc],
            text=cs[lc].apply(lambda v: str(v) if v > 0 else ''),
            textposition='inside',
        ))

    fig.update_layout(
        title=f'Artifact Lifecycle Classification — All {temporal.total_classified} Artifacts',
        barmode='stack',
        xaxis_title='Artifact count',
        height=450, width=900,
        legend=dict(orientation='h', yanchor='bottom', y=1.02),
    )
    fig_lifecycle_bars = fig
    fig.show()
else:
    print("No temporal data — skipping lifecycle bars.")

### Chart 10: Lifecycle by Creation Period

In [ ]:
fig_lifecycle_evolution = None
if 'temporal' in dir() and temporal.has_timeseries and not temporal.artifact_lifecycles.empty:
    # Filter to artifacts with commit history (they have a first_commit date)
    with_history = temporal.artifact_lifecycles[
        temporal.artifact_lifecycles['lifecycle'] != 'no-history'
    ].copy()

    if not with_history.empty:
        # Group by creation month (first_commit)
        with_history['creation_month'] = (
            with_history['first_commit']
            .dt.tz_localize(None)
            .dt.to_period('M')
            .astype(str)
        )

        # Pivot: creation_month × lifecycle
        cohort = with_history.groupby(['creation_month', 'lifecycle']).size().unstack(fill_value=0)

        lifecycle_types = ['steady', 'burst', 'set-and-forget', 'abandoned']
        lifecycle_colors = {
            'steady': '#22c55e',
            'burst': '#eab308',
            'set-and-forget': '#64748b',
            'abandoned': '#ef4444',
        }

        fig = go.Figure()
        for lc in lifecycle_types:
            if lc in cohort.columns:
                fig.add_trace(go.Bar(
                    x=cohort.index.tolist(),
                    y=cohort[lc],
                    name=lc,
                    marker_color=lifecycle_colors[lc],
                    text=cohort[lc].apply(lambda v: str(v) if v > 0 else ''),
                    textposition='inside',
                ))

        fig.update_layout(
            title=f'Lifecycle by Creation Period — {len(with_history)} Artifacts with History',
            barmode='stack',
            xaxis_title='First commit month',
            yaxis_title='Artifact count',
            height=400, width=900,
            legend=dict(orientation='h', yanchor='bottom', y=1.02),
            xaxis=dict(tickangle=-45),
        )
        fig_lifecycle_evolution = fig
        fig.show()
    else:
        print("No artifacts with commit history — skipping creation-cohort chart.")
else:
    print("No temporal data — skipping creation-cohort chart.")

### Chart 11: Author Diversity by Category

In [ ]:
fig_author_diversity = None
if 'temporal' in dir() and temporal.has_timeseries and temporal.author_diversity:
    from src.temporal_health import GROUNDING_CATEGORIES

    cats = sorted(temporal.author_diversity.keys(),
                  key=lambda c: CATEGORY_TO_LEVEL.get(c, 2))
    counts = [temporal.author_diversity[c] for c in cats]
    labels = [f"{c} (L{CATEGORY_TO_LEVEL.get(c, '?')})" for c in cats]

    # Color single-author grounding categories red
    bar_colors = []
    for c, count in zip(cats, counts):
        if count <= 1 and c in GROUNDING_CATEGORIES:
            bar_colors.append('#ef4444')  # red — bus-factor risk
        else:
            bar_colors.append('#3b82f6')  # blue — normal

    fig = go.Figure(go.Bar(
        y=labels, x=counts, orientation='h',
        marker_color=bar_colors,
        text=counts, textposition='auto',
    ))
    fig.update_layout(
        title='Author Diversity by Category (unique contributors)',
        xaxis_title='Unique authors',
        height=350, width=700,
        xaxis=dict(dtick=1),
    )
    fig_author_diversity = fig
    fig.show()
else:
    print("No temporal data — skipping author diversity chart.")

## Phase 4: LLM Report Generation

Send the structured findings to Claude to generate a narrative report.
Requires `ANTHROPIC_API_KEY` environment variable.

In [ ]:
if not os.environ.get("ANTHROPIC_API_KEY"):
    print("ANTHROPIC_API_KEY not set -- skipping LLM report generation.")
    print("Set it with: os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-...'")
    llm_report = None
else:
    import anthropic

    client = anthropic.Anthropic()

    # Build the structured findings for the prompt
    report_data = generate_report(score)

    # Collect top artifacts per level
    top_artifacts = {}
    if fc_df is not None and not fc_df.empty:
        for lvl in (2, 3, 4):
            lvl_files = fc_df[fc_df['assigned_maturity_level'] == lvl]
            top_artifacts[f'L{lvl}'] = lvl_files[['artifact_path', 'assigned_category']].head(10).to_dict('records')

    # Compute per-tool artifact counts and sample file paths for context
    tool_details = {}
    if fc_df is not None and not fc_df.empty:
        known = fc_df[fc_df['tool_name'] != 'unknown']
        for tool_name, group in known.groupby('tool_name'):
            tool_details[tool_name] = {
                'count': len(group),
                'sample_paths': group['artifact_path'].head(10).tolist() if 'artifact_path' in group.columns else [],
                'categories': sorted(group['assigned_category'].dropna().unique().tolist()),
            }

    # Build temporal health context for the LLM
    temporal_context = ""
    if 'temporal' in dir() and temporal.has_timeseries:
        temporal_context = "\n## Temporal Health Analysis\n\n"
        temporal_context += f"**Horizon date**: {temporal.horizon_date}\n\n"

        # Health verdicts
        if temporal.health_verdicts:
            temporal_context += "**Health Verdicts** (per-category lifecycle assessment):\n"
            for v in temporal.health_verdicts:
                temporal_context += f"- **{v['category']}** ({v['tier']}): dominant pattern = {v['dominant_lifecycle']}, verdict = **{v['verdict']}** — {v['message']}\n"
            temporal_context += "\n"

        # Category summaries
        if not temporal.category_summaries.empty:
            temporal_context += "**Lifecycle Distribution by Category**:\n"
            for _, row in temporal.category_summaries.iterrows():
                temporal_context += (
                    f"- {row['category']}: {row['total_artifacts']} artifacts, "
                    f"{row['total_commits']} commits — "
                    f"steady={row['steady']}, burst={row['burst']}, "
                    f"set-and-forget={row['set-and-forget']}, abandoned={row['abandoned']}\n"
                )
            temporal_context += "\n"

        # Author diversity
        if temporal.author_diversity:
            temporal_context += "**Author Diversity** (unique contributors per category):\n"
            for cat, count in sorted(temporal.author_diversity.items()):
                temporal_context += f"- {cat}: {count} unique author(s)\n"
            temporal_context += "\n"

        temporal_context += """**Temporal health rules**: Grounding artifacts (rules, architecture, code-style, configuration) should be living documents — "set-and-forget" or "abandoned" is a concern. Agentic artifacts (agents, commands, skills) are OK if stable. Session-logs are expected to have burst patterns. Single-author grounding categories are a bus-factor risk.\n"""

    prompt = f"""You are an AI adoption analyst for an anonymous benchmark study.

Analyze the following maturity assessment for repository "{REPO_NAME}" and generate a report.

## Assessment Data

**Overall Level**: L{report_data['overall_level']} -- {report_data['overall_label']}
**Confidence**: {report_data['confidence']:.0%}
**Total Artifacts**: {report_data['artifact_count']}
**Tools Detected**: {', '.join(report_data['tools_detected']) or 'none'}

**Per-tool details** (count + sample file paths):
{json.dumps(tool_details, indent=2)}

**Level Evidence**:
{json.dumps(report_data['level_evidence'], indent=2)}

**Category Counts**:
{json.dumps(report_data['category_counts'], indent=2)}

**Coherence Flags**:
{json.dumps(report_data['coherence_flags'], indent=2)}

**Signal Agreement Rate**: {report_data['signal_agreement_rate']:.0%}
**Category Concentration**: {report_data['category_concentration']:.2f}

**Top Artifacts per Level**:
{json.dumps(top_artifacts, indent=2)}
{temporal_context}
## IMPORTANT CONSTRAINTS

1. **There are exactly 4 maturity levels: L1, L2, L3, L4.** There is NO L5. Do not reference any level beyond L4.

2. **Recommendations must be practical and actionable from a developer's perspective.** Focus on what the developer should DO with their AI tools -- e.g., "create agent definitions to delegate code review to AI" or "add workflow orchestration files to coordinate multi-step tasks." Do NOT recommend things like "improve signal consistency", "increase category balance", or other framework-internal metrics -- those are meaningless to the end user.

3. **Carefully verify tool attributions before making recommendations about specific tools.** Look at the ACTUAL FILE PATHS for each tool. If a "tool" has very few artifacts and those file paths look like standard repository files (e.g., `.github/workflows/`, `.github/ISSUE_TEMPLATE/`, `README.md`) rather than actual AI tool configurations, that tool is likely a FALSE POSITIVE from pattern matching. Do NOT recommend expanding or integrating a tool that is actually a false positive. Instead, note that it was likely misattributed.

4. **This is a single-repository assessment.** Do not recommend "balancing" between tools or "integrating" tools unless the repository genuinely uses multiple AI tools. If only one tool is legitimately present, focus recommendations on deepening that tool's adoption.

5. **Do not recommend actions the repository already does well.** If all coherence checks pass and the repo is at L4, focus on optimization and maintenance rather than "add more of X."

6. **If temporal health data is provided, incorporate it into your analysis.** Comment on artifact staleness risks, bus-factor concerns (single-author grounding categories), and whether lifecycle patterns match expectations for each category tier. Flag any grounding artifacts that are "set-and-forget" or "abandoned" as maintenance risks.

## Instructions

Generate a structured report with:
1. **Executive Summary** (2-3 sentences about the repository's AI adoption maturity)
2. **Evidence Analysis** -- Why this repo is at this maturity level, with specific evidence citations from artifact paths
3. **Temporal Health** -- If temporal data is available, analyze artifact maintenance patterns, staleness risks, and contributor diversity. If no temporal data, note its absence.
4. **Recommendations** -- Practical, actionable steps the developer can take to improve their AI tool setup. Each recommendation should clearly explain the benefit.
5. **Notable Observations** -- Interesting patterns (e.g., tool dominance, embedded grounding, category distribution, lifecycle patterns)

Be specific and cite artifact paths where relevant. Use markdown formatting.
"""

    response = client.messages.create(
        model="claude-sonnet-4-5-20250929",
        max_tokens=2000,
        messages=[{"role": "user", "content": prompt}],
    )

    llm_report = response.content[0].text
    print(llm_report)

## Phase 5: Batch Mode

Score all repositories in the output directory.

In [ ]:
# Discover all repos
repos = []
for item in OUTPUT_DIR.iterdir():
    if item.is_dir() and not item.name.startswith('.'):
        if list(item.glob('*_file_artifacts.csv')):
            repos.append(item.name)
repos = sorted(repos)
print(f"Found {len(repos)} repositories")

# Score each repo
batch_results = []
for repo in repos:
    try:
        s = score_from_output_dir(
            str(OUTPUT_DIR), repo, model,
            artifacts_dir=ARTIFACTS_DIR,
        )
        batch_results.append({
            'repo': repo,
            'level': s.overall_level,
            'label': s.overall_label,
            'confidence': s.confidence,
            'artifacts': s.artifact_count,
            'tools': ', '.join(s.tools_detected),
            'l2_primary': s.level_evidence.get(2, {}).get('primary', 0),
            'l3_primary': s.level_evidence.get(3, {}).get('primary', 0),
            'l4_primary': s.level_evidence.get(4, {}).get('primary', 0),
        })
        print(f"  {repo}: L{s.overall_level} ({s.confidence:.0%}) — {s.artifact_count} artifacts")
    except Exception as e:
        print(f"  {repo}: ERROR — {e}")
        batch_results.append({'repo': repo, 'level': 0, 'label': 'Error', 'confidence': 0})

batch_df = pd.DataFrame(batch_results)
display(batch_df.sort_values('level', ascending=False))

In [ ]:
# Batch level distribution
if not batch_df.empty:
    level_dist = batch_df['level'].value_counts().sort_index()
    all_levels = [1, 2, 3, 4]
    counts = [level_dist.get(lvl, 0) for lvl in all_levels]
    labels = ['L1 Ad Hoc', 'L2 Grounded', 'L3 Agent-Augmented', 'L4 Orchestration']
    colors = ['#6b7280', '#3b82f6', '#f97316', '#22c55e']

    fig = go.Figure(go.Bar(
        x=labels, y=counts,
        marker_color=colors,
        text=counts, textposition='auto',
    ))
    fig.update_layout(
        title=f'Maturity Level Distribution ({len(batch_df)} repositories)',
        yaxis_title='Repository count',
        height=400, width=700,
    )
    fig.show()

## Phase 6: Export

In [ ]:
# Export single-repo report as JSON
report_data = generate_report(score)
report_data['repo_name'] = REPO_NAME
report_data['timestamp'] = datetime.now().isoformat()
report_data['model'] = DEFAULT_MODEL

report_path = EXPORT_DIR / f'{REPO_NAME}_maturity_report.json'
with open(report_path, 'w') as f:
    json.dump(report_data, f, indent=2, default=str)
print(f"Exported: {report_path}")

# Export batch results CSV
if not batch_df.empty:
    batch_path = EXPORT_DIR / 'maturity_scores_batch.csv'
    batch_df.to_csv(batch_path, index=False)
    print(f"Exported: {batch_path}")

# Export file classifications CSV
if fc_df is not None and not fc_df.empty:
    fc_path = EXPORT_DIR / f'{REPO_NAME}_file_classifications.csv'
    fc_df.to_csv(fc_path, index=False)
    print(f"Exported: {fc_path}")

## Phase 7: PDF Report

Generate a self-contained executive PDF report with all charts and descriptions.

In [ ]:
import importlib
import src.report_generator
importlib.reload(src.report_generator)
from src.report_generator import generate_pdf_report

figures = {
    'gauge': fig_gauge,
    'stacking': fig_stacking,
    'categories': fig_categories,
    'agreement': fig_agreement,
    'sunburst': fig_sunburst,
    'tool_category': fig_tool_cat,
    'hybrid': fig_hybrid,
    'coherence': fig_coherence,
    'lifecycle_bars': fig_lifecycle_bars if 'fig_lifecycle_bars' in dir() else None,
    'lifecycle_evolution': fig_lifecycle_evolution if 'fig_lifecycle_evolution' in dir() else None,
    'author_diversity': fig_author_diversity if 'fig_author_diversity' in dir() else None,
}

pdf_path = generate_pdf_report(
    score=score,
    repo_name=REPO_NAME,
    figures=figures,
    output_path=str(EXPORT_DIR / f'{REPO_NAME}_maturity_report.pdf'),
    llm_report=llm_report if 'llm_report' in dir() and llm_report else None,
    temporal_health=temporal if 'temporal' in dir() and temporal.has_timeseries else None,
)
print(f"PDF report exported: {pdf_path}")

In [ ]:
# Unload model
print("Unloading model...")
del model
gc.collect()
print("Done.")